# Analyze Drudge hyperlinks

In [161]:
import re
import sys
from pathlib import Path
from urllib.parse import urlparse

In [119]:
import storysniffer

In [120]:
import numpy as np
import pandas as pd
import altair as alt

In [121]:
this_dir = Path("__file__").parent.absolute()
sys.path.append(this_dir.parent)
sys.path.append(str(this_dir.parent / "newshomepages"))

In [122]:
import altair_theme

In [123]:
alt.themes.register('palewire', altair_theme.theme)
alt.themes.enable('palewire')

ThemeRegistry.enable('palewire')

In [124]:
extracts_dir = this_dir.parent / "extracts" / "csv"

In [125]:
analysis_dir = this_dir.parent / "_analysis"

Read in the sample data

In [126]:
df = pd.read_csv(
    extracts_dir / "drudge-hyperlinks-sample.csv",
    usecols=[
        'handle',
        'file_name',
        'date',
        'text',
        'url',
    ],
    dtype=str,
    parse_dates=["date"]
)

In [127]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56426 entries, 0 to 56425
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   handle     56426 non-null  object        
 1   file_name  56426 non-null  object        
 2   date       56426 non-null  datetime64[ns]
 3   text       56182 non-null  object        
 4   url        56425 non-null  object        
dtypes: datetime64[ns](1), object(4)
memory usage: 2.2+ MB


In [128]:
df.head()

,handle,file_name,date,text,url
0,DRUDGE,drudge-2022-05-21T14:41:56.234274-04:00.hyperl...,2022-05-21,"MUSK BLASTS BIDEN, DEMS OVER 'HATE'...",https://www.cnbc.com/2022/05/20/tesla-boss-elo...
1,DRUDGE,drudge-2022-05-21T14:41:56.234274-04:00.hyperl...,2022-05-21,STOCK -44% FOR YEAR...,https://www.google.com/finance/quote/TSLA:NASD...
2,DRUDGE,drudge-2022-05-21T14:41:56.234274-04:00.hyperl...,2022-05-21,'THERE WILL BE BLOOD'...,https://www.rawstory.com/elon-musk-tesla-stock/
3,DRUDGE,drudge-2022-05-21T14:41:56.234274-04:00.hyperl...,2022-05-21,PRESSURE INTENSIFIES ON WORLD'S RICHEST MAN...,https://www.msn.com/en-us/finance/savingandinv...
4,DRUDGE,drudge-2022-05-21T14:41:56.234274-04:00.hyperl...,2022-05-21,TESLA's golden moment over?,https://unherd.com/thepost/teslas-golden-momen...


Guess links with `storysniffer`

In [129]:
links_df = df.groupby(["text", "url"]).size().rename("n").reset_index()

In [130]:
sniffer = storysniffer.StorySniffer()

/home/palewire/.local/share/virtualenvs/news-homepages-Qlfa7zLV/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator CountVectorizer from version 1.1.1 when using version 1.1.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/palewire/.local/share/virtualenvs/news-homepages-Qlfa7zLV/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator FunctionTransformer from version 1.1.1 when using version 1.1.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/palewire/.local/share/virtualenvs/news-homepages-Qlfa7zLV/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying t

In [131]:
links_df.describe()

,n
count,6779.000000
mean,8.287506
std,36.701769
min,1.000000
25%,1.000000
50%,2.000000
75%,3.000000
max,232.000000


In [132]:
links_df['is_story'] = links_df.apply(lambda x: sniffer.guess(x['url'], text=x['text']), axis=1)

In [133]:
links_df.is_story.value_counts()

True     6511
False     268
Name: is_story, dtype: int64

In [134]:
links_df.is_story.value_counts(normalize=True)

True     0.960466
False    0.039534
Name: is_story, dtype: float64

In [135]:
links_df.to_csv("./drudge-hyperlinks-storysniffer-guesses.csv", index=False)

Make some manual fixes

In [136]:
blacklist = [
    "/privacy/",
]

In [137]:
links_df.loc[
    links_df.url.isin(blacklist),
    'is_story'
] = False

In [138]:
substack_pattern = ".(substack|theankler|commonsense).(com|news)/p/"

In [139]:
links_df.loc[
    links_df.url.str.contains(substack_pattern),
    'is_story'
] = True

/tmp/ipykernel_564195/673093257.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  links_df.url.str.contains(substack_pattern),


In [140]:
time_pattern = "^https://time.com/\d{5,}/*"

In [141]:
links_df.loc[
    links_df.url.str.contains(time_pattern),
    'is_story'
] = True

In [142]:
studyfinds_pattern = "^https://www.studyfinds.org/*.{5,}"

In [143]:
links_df.loc[
    links_df.url.str.contains(studyfinds_pattern),
    'is_story'
] = True

In [144]:
bbc_pattern = "^https://www.bbc.com/news/*.{5,}"

In [145]:
links_df.loc[
    links_df.url.str.contains(bbc_pattern),
    'is_story'
] = True

In [146]:
jpost_pattern = "^https://www.jpost.com/breaking-news/*.{5,}"

In [147]:
links_df.loc[
    links_df.url.str.contains(jpost_pattern),
    'is_story'
] = True

In [154]:
braint_pattern = "^https://www.braintomorrow.com/*.{5,}"

In [155]:
links_df.loc[
    links_df.url.str.contains(braint_pattern),
    'is_story'
] = True

Knock out anything that appears most of the time

In [156]:
n = len(df.file_name.unique())

In [157]:
n * .5

116.0

In [158]:
links_df.loc[
    links_df.n >= n,
    'is_story',
] = False

In [159]:
links_df.to_csv("./drudge-hyperlinks-storysniffer-tweaks.csv", index=False)

In [160]:
links_df[
    (links_df.n < 40) &
    (links_df.is_story == False)
].sort_values("url").head(50)

,text,url,n,is_story
3326,LIVE: TEMP MAP,http://hp2.wright-weather.com/icons/us_heat.gif,2,False
3321,LIVE HEAT MAP...,http://hp2.wright-weather.com/icons/us_heat.gif,8,False
3327,LIVE: TEMP MAP...,http://hp2.wright-weather.com/icons/us_heat.gif,14,False
697,BAZ BAMIGBOYE,http://www.dailymail.co.uk/tvshowbiz/columnist...,1,False
3986,NOW JILL HAS COVID...,https://abcnews.go.com/US/lady-jill-biden-test...,2,False
4293,PARTICIPANTS...,https://bilderbergmeetings.org/press/press-rel...,1,False
1643,DEPP V. HEARD: $50M Defamation Case Goes To Ju...,https://deadline.com/tag/depp-heard-trial/,1,False
1641,DEPP JUSTICE!,https://deadline.com/tag/depp-heard-trial/,1,False
3486,MAP...,https://earthquake.usgs.gov/earthquakes/eventp...,1,False
3487,MAP...,https://earthquake.usgs.gov/earthquakes/eventp...,5,False


Tally domains

In [162]:
links_df['domain'] = links_df.url.apply(lambda x: urlparse(x).netloc)

In [168]:
links_df[links_df.is_story].domain.value_counts().reset_index().head(25)

,index,domain
0,www.msn.com,756
1,apnews.com,488
2,www.wsj.com,367
3,news.yahoo.com,317
4,dnyuz.com,282
5,www.dailymail.co.uk,267
6,www.cnbc.com,204
7,www.cnn.com,202
8,thehill.com,200
9,www.the-sun.com,199
